In [ ]:
# Connecting to MongoDB
from pymongo import MongoClient
client = MongoClient('localhost', 27017)


In [ ]:
# Going into the place where we need to work on
db = client["db_to_work_on"]
coll = db["collection_to_work_on"]
# If the db or coll is not present in mongodb, it will be automatically created

In [ ]:
# Initialising objects
import json 
#import csv if it is a csv file


with open('file_to_insert.json') as jsonfile: # NEEDED COMMENT: file closes automatically
    documents = json.load(jsonfile) 
    #documetns = csv.load(csvfile) if the file is csv
    result = coll.insert_many(documents)    

In [ ]:
# Counting objects
coll.count_documents({})

In [ ]:
# Finding Objects
query = {
         'title': {'$exists': True}, # If title does not exist, the document will not be outputted
         'pages': {'$gt': 500}, # If pages is not Greater than 500, the document will not be outputted
          # Similar ones include: $lt - Less than, $sq - Equals to

         '$and': { 
                  'name': {'$in': ['abc','edf'] }, 
                  'sex': 'Male' 
                  # When we use $in, it means that any value inside the array is possible. 
                  # Without in, it has to exactly match the value, in this case 'sex' == 'Male'

                 }, # Only when both conditions are met will the document be outputted 
         'desc': {'$regex': 'blahblahblah'} # regex will return output if the desc contains the words ANYWHERE within the string/list/array
        }

print(query)

proj = {'_id': 0, 'title': 1, 'author': 1}
# 0 indicates to hide the column, 1 will show the column. The default is 0
# _id will always be displayed by default, so there is a need to specifically state that it is 0 to hide it

elem_to_sort = [('title', 1), ('idx', -1)] # All the elements inside here are in the format of tuples ()
# The first tuple in the array will be sorted, then if there are any repeats, they will be sorted by the second and so on
# 1 for ascending, -1 for descending

max_to_display = 10 # This will filter out the top 10 elements of the documents in the collection that we need

for doc in coll.find(query, proj).sort(elem_to_sort).limit(max_to_display):
    print(doc)

In [ ]:
# Inserting one object
doc_to_insert = {'name', 'yes', 'title', 'blah'}

result = coll.insert_one(doc_to_insert) # result is needed to hold the id of the inserted documents, needed for referencing and debuggin

# To see the inserted object,
query = {"_id": result.inserted_id}

inserted_doc = coll.find_one(query) # Without stating the projection, it is taken as proj = {}
print(inserted_doc)

In [ ]:
# Inserting many objects
list_of_doc_to_insert = [
    {'name', 'yes1', 'title', 'blaha'},
    {'name', 'yes2', 'title', 'blahb'},
    {'name', 'yes3', 'title', 'blahc'}
]

results = coll.insert_many(list_of_doc_to_insert) # result is needed to hold the id of the inserted documents, needed for referencing and debuggin

# To see the inserted object,
query = {"_id": {"$in": result.inserted_ids} }

inserted_docs = coll.find() # Without stating the projection, it is taken as proj = {}
for doc in inserted_docs:
    print(doc)

In [ ]:
# Updating one object
query = {'title': 'alpha or nah'} # Titles should usually be more or less unique,so updating this will only update one document
newval = {"$set":{"title":"beta"} }

result = coll.update_one(query, newval)# Result is needed to hold the id of the inserted documents, needed for referencing and debuggin

updated_doc = coll.find_one(query) #Without stating the projection, it is taken as proj = {}
print(updated_doc)

In [ ]:
# Updating many objects
query = {'sex': 'Male'} # Many objects may have this, so updating this query will result in many updates being made
newval = {'$set': {'sex': 'Gay'} }

result = coll.update_many(query, newval)

print(result.modified_count)

In [ ]:
# Deleting Objects
query = {'isbn': '1023495'} #Once again, isbn is basically unique for every book, so this query will only access one document
result = coll.delete_one(query)

In [ ]:
query = {"$or": [
                 {"pageCount": 0}, 
                 {"isbn": {"$exists": False} }
                ]
        } # Here, there can be many document with the same values, therefore delete many.

result = coll.delete_many(query)